# Migration des Workspaces Fabric vers Runtime 2.0

## Objectif

Migrer l'ensemble des Workspaces d'un tenant vers **Fabric Runtime 2.0**, Runtime 1.2 étant hors support depuis le **31 mars 2026**.

Le notebook procède en trois temps : **inventorier** tout le tenant, **mesurer l'impact** pour distinguer ce qui peut être migré sans risque de ce qui exige une validation, puis **migrer** et produire un **compte rendu**.

## Pourquoi le rôle Fabric Administrator ne suffit pas

Les API Fabric sont asymétriques. La découverte est tenant-wide, la modification ne l'est pas.

| API | Portée | Permission |
|---|---|---|
| `GET /v1/admin/workspaces` | **Tenant** | Fabric Administrator, `Tenant.Read.All` |
| `GET /v1/admin/items` | **Tenant** | Fabric Administrator, `Tenant.Read.All` |
| `GET /v1/workspaces/{id}/spark/settings` | **Workspace** | rôle *Viewer* minimum |
| `PATCH /v1/workspaces/{id}/spark/settings` | **Workspace** | rôle *Admin* |
| `.../environments/{id}/staging/sparkcompute` | **Item** | écriture sur l'Environment |

Vous voyez donc 100 % du tenant, mais ne pouvez migrer que les Workspaces où vous détenez un rôle. Trois réponses possibles :

1. **`AUTO_ELEVATE = True`** — le notebook s'attribue le rôle *Admin*, migre, puis se retire.
2. **Groupe de sécurité** ajouté comme Admin sur les Workspaces concernés — préférable en gouvernance durable.
3. **Portail d'administration** → *Workspaces* → *Access*, pour un traitement ponctuel.

Les Workspaces restés inaccessibles ne sont jamais masqués : ils apparaissent comme `Forbidden` dans l'inventaire et dans le compte rendu.

## Segmentation de l'impact

Le changement de Runtime n'affecte que les charges **Spark**. Un Workspace qui ne contient ni Notebook, ni Spark Job Definition, ni Environment n'exécute aucun code Spark : sa migration est une opération de configuration sans effet fonctionnel.

| Vague | Contenu | Risque | Traitement |
|---|---|---|---|
| **A** | Aucun Notebook, SJD ni Environment — typiquement des rapports Power BI | Négligeable | Migration directe |
| **B** | Au moins un Notebook, SJD ou Environment | Réel | Prévenir les Admins, tester, puis migrer |

> Un Lakehouse sans Notebook reste en vague A : le Runtime appliqué est celui du Workspace **qui exécute** le code, pas celui qui héberge les données.

## Ce qui change entre les Runtimes

| Composant | Runtime 1.2 | Runtime 1.3 | Runtime 2.0 |
|---|---|---|---|
| Statut | Hors support | GA jusqu'au 30/09/2026 | GA |
| Fin de support | 31 mars 2026 | mars 2027 | 31 août 2028 |
| Apache Spark | 3.4.1 | 3.5 | 4.1 |
| Java | 11 | 11 | 21 |
| Scala | 2.12.17 | 2.12.17 | 2.13 |
| Python | 3.10 | 3.11 | 3.13 |
| Delta Lake | 2.4 | 3.2 | 4.2 |

## Points de vigilance sur les Environments

Microsoft signale deux comportements à anticiper :

- **Les bibliothèques sont migrées automatiquement**, mais les **JAR** ont une probabilité significative de ne plus fonctionner du fait des changements de Scala, Java et Spark. En cas de conflit, **la publication échoue** — le notebook traite ce cas et le remonte dans le compte rendu.
- Après la mise à jour du socle Python, certains Environments renvoient `LibraryManagementError: An upgrade to the base Spark Python environment has been detected`. Correctif : retirer toutes les bibliothèques, publier, les réajouter, publier à nouveau.

## Déroulé recommandé

1. Exécuter les sections 0 à 4 en `DRY_RUN` pour obtenir l'inventaire, la segmentation et la liste de diffusion.
2. Migrer la **vague A** (`MIGRATION_WAVE = "A"`) — sans risque fonctionnel.
3. Prévenir les Admins de la **vague B** à partir de `notification_df`.
4. Migrer la vague B par lots après validation, en renseignant `TARGET_WORKSPACES`.
5. Archiver le compte rendu de la section 6.


# 0. Paramètres et utilitaires

In [ ]:
import re
import time
import json

from urllib.parse import urlencode

import pandas as pd
import sempy.fabric as fabric

# ============================================================
# CIBLE
# ============================================================

TARGET_RUNTIME  = "2.0"                     # runtime GA le plus recent
SOURCE_RUNTIMES = {"1.1", "1.2", "1.3"}     # runtimes consideres comme a migrer

# ============================================================
# PERIMETRE
# ============================================================

INCLUDE_PERSONAL  = False   # inclure les "My workspaces"
TARGET_WORKSPACES = []      # [] = tout le tenant, sinon une liste de noms

# Vague de migration :
#   "A"   workspaces sans charge Spark  (ni notebook, ni SJD, ni environment)
#   "B"   workspaces avec charge Spark  (validation applicative requise)
#   "ALL" les deux
MIGRATION_WAVE = "A"

# ============================================================
# GARDE-FOUS
# ============================================================

DRY_RUN         = True    # True = plan seul, aucune ecriture
CONFIRMATION    = ""      # doit valoir exactement f"UPGRADE TO {TARGET_RUNTIME}"
BLOCK_DOWNGRADE = True    # refuse de rétrograder un runtime plus recent que la cible

# ============================================================
# ELEVATION JUST-IN-TIME
# ============================================================

AUTO_ELEVATE           = False   # s'attribuer le role Admin le temps de la migration
ELEVATE_PRINCIPAL      = None    # UPN (User) ou objectId (App / Group)
ELEVATE_PRINCIPAL_TYPE = "User"  # "User" | "App" | "Group"
REVOKE_AFTER_MIGRATION = True
ELEVATION_WAIT_S       = 30      # propagation du role avant le premier appel

# ============================================================
# OPTIONS
# ============================================================

RESOLVE_CONTACTS    = True    # resoudre les Admins a prevenir (vague B)
NOTIFY_ROLES        = ["Admin", "Member"]   # ajouter "Contributor" pour toucher les auteurs de notebooks
DEEP_SCAN_NOTEBOOKS = False   # lire la definition des notebooks (Environment attache)
MAX_ADMIN_CALLS     = 180     # garde-fou sous la limite Microsoft de 200 appels/heure

# ============================================================
# CONTROLES DE COHERENCE
# ============================================================

if TARGET_RUNTIME in SOURCE_RUNTIMES:
    raise ValueError("TARGET_RUNTIME ne peut pas figurer dans SOURCE_RUNTIMES.")

if MIGRATION_WAVE not in ("A", "B", "ALL"):
    raise ValueError("MIGRATION_WAVE doit valoir A, B ou ALL.")

if ELEVATE_PRINCIPAL_TYPE not in ("User", "App", "Group"):
    raise ValueError(
        f"ELEVATE_PRINCIPAL_TYPE doit valoir User, App ou Group. Recu : {ELEVATE_PRINCIPAL_TYPE!r}. "
        "L'UPN ou l'objectId se renseigne dans ELEVATE_PRINCIPAL, pas ici."
    )

if AUTO_ELEVATE and not ELEVATE_PRINCIPAL:
    raise ValueError(
        "AUTO_ELEVATE = True mais ELEVATE_PRINCIPAL est vide. "
        "Renseigner l'UPN pour un User, ou l'objectId pour une App ou un Group."
    )

EXPECTED_CONFIRMATION = f"UPGRADE TO {TARGET_RUNTIME}"
WRITE_ENABLED = (not DRY_RUN) and CONFIRMATION == EXPECTED_CONFIRMATION

client = fabric.FabricRestClient()

_admin_calls = 0


# ============================================================
# COUCHE HTTP
# ============================================================


class _ErrorResponse:
    """FabricRestClient leve sur un code non-2xx : on reconstruit une reponse exploitable."""

    def __init__(self, status_code, text="", headers=None):
        self.status_code = status_code
        self.text = text
        self.headers = headers or {}

    def json(self):
        try:
            return json.loads(self.text)
        except Exception:
            return {}


def _as_response(exception):
    inner = getattr(exception, "response", None)

    if inner is not None and hasattr(inner, "status_code"):
        return inner

    text = str(exception)
    status = getattr(exception, "status_code", None)

    if not isinstance(status, int):
        match = re.search(r"\b([45]\d{2})\b", text)
        status = int(match.group(1)) if match else 0

    return _ErrorResponse(status, text)


def _url(path, params=None):
    params = {k: v for k, v in (params or {}).items() if v is not None}
    return f"{path}?{urlencode(params)}" if params else path


def api_call(method, path, params=None, json_body=None, max_retries=4, is_admin=False):
    """Appel REST. Renvoie toujours un objet reponse, jamais une exception HTTP."""
    global _admin_calls

    if is_admin:
        if _admin_calls >= MAX_ADMIN_CALLS:
            raise RuntimeError(
                f"Garde-fou atteint : {_admin_calls} appels admin. "
                "La limite Microsoft est de 200 par heure."
            )
        _admin_calls += 1

    url = _url(path, params)
    fn = getattr(client, method)

    for attempt in range(max_retries + 1):
        try:
            response = fn(url, json=json_body) if json_body is not None else fn(url)
        except Exception as ex:
            response = _as_response(ex)

        if response.status_code == 429 and attempt < max_retries:
            wait = int(response.headers.get("Retry-After", 30))
            print(f"  429 throttling, pause {wait}s ({path})")
            time.sleep(min(wait, 300))
            continue

        return response

    return response


# Causes reelles observees sur un tenant : toutes ne se resolvent pas par un role.
STATUS_ACTIONS = {
    "Ok":                  ("Runtime lisible", "Aucune"),
    "Forbidden":           ("Aucun role sur le workspace",
                            "Attribuer un role : AUTO_ELEVATE, groupe Viewer ou portail"),
    "NoFabricCapacity":    ("Aucune capacite Fabric assignee au workspace",
                            "Hors perimetre tant qu'aucune capacite n'est assignee"),
    "CapacityUnavailable": ("Capacite Fabric assignee mais inactive",
                            "Reprendre la capacite puis relancer l'inventaire"),
    "Throttled":           ("Quota d'appels depasse", "Attendre puis relancer"),
}

# L'errorCode est stable, contrairement au code HTTP qui recouvre plusieurs causes.
ERROR_CODE_STATUS = {
    "WorkspaceHasNoCapacityAssigned": "NoFabricCapacity",
    "CapacityNotActive":              "CapacityUnavailable",
    "InsufficientPrivileges":         "Forbidden",
    "Unauthorized":                   "Forbidden",
    "EntityNotFound":                 "NotFound",
}

HTTP_CODE_STATUS = {401: "Forbidden", 403: "Forbidden", 429: "Throttled"}


def describe_status(response):
    """Classe un echec par errorCode Fabric, avec repli sur le code HTTP."""
    code = response.status_code

    if code == 200:
        return "Ok", None

    try:
        body = response.json() or {}
    except Exception:
        body = {}

    error_code = body.get("errorCode") or ""
    message = (body.get("message") or response.text or "")[:160]

    status = ERROR_CODE_STATUS.get(error_code) or HTTP_CODE_STATUS.get(code) or f"HTTP{code}"

    return status, (f"{error_code} : {message}" if error_code else message)


def _extract_list(body):
    """Les API Fabric n'exposent pas toutes leur collection sous la meme cle."""
    for key in ("value", "workspaces", "itemEntities", "items", "accessDetails", "data"):
        if isinstance(body.get(key), list):
            return body[key]

    for value in body.values():
        if isinstance(value, list):
            return value

    return []


def get_paginated(path, params=None, is_admin=False, max_pages=100):
    """Suit continuationToken jusqu'a epuisement de la collection."""
    collected, token, pages = [], None, 0

    while pages < max_pages:
        query = dict(params or {})

        if token:
            query["continuationToken"] = token

        response = api_call("get", path, params=query, is_admin=is_admin)

        if response.status_code != 200:
            raise RuntimeError(f"{path} -> {response.status_code} : {response.text[:400]}")

        body = response.json()
        collected.extend(_extract_list(body))

        token = body.get("continuationToken")
        pages += 1

        if not token:
            break

    return collected


def admin_list_items(item_type):
    """Liste tenant-wide d'un type d'item, avec repli si le filtre serveur est refuse."""
    try:
        return get_paginated("/v1/admin/items", {"type": item_type}, is_admin=True)

    except RuntimeError as ex:
        print(f"  Filtre type={item_type} refuse ({ex}). Repli sur un listing complet.")
        every_item = get_paginated("/v1/admin/items", is_admin=True)
        return [i for i in every_item if str(i.get("type", "")).lower() == item_type.lower()]


def wait_for_operation(response, timeout_s=1800, label=""):
    """Suit une operation longue jusqu'a son etat terminal. Un 202 ne prouve pas le succes."""
    if response.status_code == 200:
        return "Succeeded", None

    if response.status_code != 202:
        return "Failed", f"HTTP {response.status_code}: {response.text[:300]}"

    operation_id = response.headers.get("x-ms-operation-id")
    delay = int(response.headers.get("Retry-After", 20))

    if not operation_id:
        return "Unknown", "202 sans en-tete x-ms-operation-id"

    deadline = time.time() + timeout_s

    while time.time() < deadline:
        time.sleep(min(delay, 60))

        poll = api_call("get", f"/v1/operations/{operation_id}")

        if poll.status_code != 200:
            return "Unknown", f"Polling HTTP {poll.status_code}"

        body = poll.json()
        status = body.get("status", "Unknown")

        if status in ("Succeeded", "Failed", "Cancelled", "Undefined"):
            error = json.dumps(body.get("error")) if body.get("error") else None
            return status, error

    return "Timeout", f"Non termine apres {timeout_s}s ({label})"


def runtime_rank(version):
    """Ordonne les versions de runtime pour detecter une retrogradation."""
    try:
        return tuple(int(part) for part in str(version).split("."))
    except (TypeError, ValueError):
        return None


TARGET_RANK = runtime_rank(TARGET_RUNTIME)

print("=" * 62)
print("CONFIGURATION")
print("=" * 62)
print(f"Runtime cible   : {TARGET_RUNTIME}")
print(f"Runtimes source : {sorted(SOURCE_RUNTIMES)}")
print(f"Vague           : {MIGRATION_WAVE}")
print(f"DRY_RUN         : {DRY_RUN}")
print(f"Ecriture        : {'ACTIVEE' if WRITE_ENABLED else 'BLOQUEE'}")
print(f"Elevation JIT   : {AUTO_ELEVATE}")

if AUTO_ELEVATE:
    print(f"  Principal     : {ELEVATE_PRINCIPAL} ({ELEVATE_PRINCIPAL_TYPE})")
    print(f"  Retrait apres : {REVOKE_AFTER_MIGRATION}")

if not WRITE_ENABLED and not DRY_RUN:
    print(f"  CONFIRMATION attendue : {EXPECTED_CONFIRMATION}")


# 1. Inventaire des Workspaces

Recense **tout le tenant** via l'Admin API, puis lit le Runtime par défaut de chaque Workspace.

La lecture du Runtime utilise une API à portée Workspace. Elle échoue pour plusieurs raisons, qui n'appellent pas la même réponse :

| `AccessStatus` | `errorCode` Fabric | Cause | Action |
|---|---|---|---|
| `Ok` | — | Runtime lisible | Aucune |
| `Forbidden` | `InsufficientPrivileges` | Aucun rôle sur le Workspace | Attribuer un rôle — c'est le **seul** cas que `AUTO_ELEVATE` résout |
| `NoFabricCapacity` | `WorkspaceHasNoCapacityAssigned` | Aucune capacité Fabric assignée au Workspace | Hors périmètre **tant qu'aucune capacité n'est assignée** : sans capacité, il n'existe pas de Runtime Spark |
| `CapacityUnavailable` | `CapacityNotActive` | Capacité assignée mais en pause | Reprendre la capacité, puis relancer l'inventaire |

Aucun de ces Workspaces n'est écarté silencieusement : ils restent dans l'inventaire avec leur cause et l'action correspondante.

In [ ]:
# ============================================================
# 1. Perimetre du tenant et runtime par defaut
# ============================================================

user_workspaces = fabric.list_workspaces()
user_workspace_ids = set(user_workspaces["Id"].astype(str))

raw_workspaces = get_paginated("/v1/admin/workspaces", is_admin=True)

all_workspaces_df = pd.DataFrame([
    {
        "WorkspaceId":   str(ws.get("id")),
        "WorkspaceName": ws.get("name") or ws.get("displayName"),
        "Type":          ws.get("type"),
        "State":         ws.get("state"),
        "CapacityId":    ws.get("capacityId"),
    }
    for ws in raw_workspaces
])

# ------------------------------------------------------------
# Entonnoir de filtrage : chaque workspace ecarte est justifie.
# ------------------------------------------------------------

all_workspaces_df["ExclusionReason"] = None

is_inactive = all_workspaces_df["State"].astype(str).str.lower() != "active"
all_workspaces_df.loc[is_inactive, "ExclusionReason"] = "State != Active"

if not INCLUDE_PERSONAL:
    is_personal = all_workspaces_df["Type"].astype(str).str.lower() == "personal"
    all_workspaces_df.loc[
        is_personal & all_workspaces_df["ExclusionReason"].isna(), "ExclusionReason"
    ] = "Personal (My workspace)"

if TARGET_WORKSPACES:
    out_of_scope = ~all_workspaces_df["WorkspaceName"].isin(TARGET_WORKSPACES)
    all_workspaces_df.loc[
        out_of_scope & all_workspaces_df["ExclusionReason"].isna(), "ExclusionReason"
    ] = "Hors TARGET_WORKSPACES"

excluded_df = all_workspaces_df[all_workspaces_df["ExclusionReason"].notna()]

workspaces_df = (
    all_workspaces_df[all_workspaces_df["ExclusionReason"].isna()]
    .drop(columns=["ExclusionReason"])
    .reset_index(drop=True)
)

workspaces_df["HasMyAccess"] = workspaces_df["WorkspaceId"].isin(user_workspace_ids)

print("=" * 62)
print("PERIMETRE")
print("=" * 62)
print(f"Retournes par l'Admin API : {len(all_workspaces_df)}")

for reason, count in excluded_df["ExclusionReason"].value_counts().items():
    print(f"  exclus, {reason:<24} : {count}")

print(f"Retenus                   : {len(workspaces_df)}")
print(f"  avec un role direct     : {int(workspaces_df['HasMyAccess'].sum())}")
print(f"  sans role direct        : {int((~workspaces_df['HasMyAccess']).sum())}")
print(f"\nPour memoire, portee utilisateur seule : {len(user_workspace_ids)}")

if len(excluded_df):
    display(excluded_df[["WorkspaceName", "Type", "State", "ExclusionReason"]])

# ------------------------------------------------------------
# Lecture du runtime par defaut (API a portee workspace).
# ------------------------------------------------------------

settings_rows = []

for _, ws in workspaces_df.iterrows():

    workspace_id = ws["WorkspaceId"]
    response = api_call("get", f"/v1/workspaces/{workspace_id}/spark/settings")
    status, detail = describe_status(response)

    runtime = None
    default_environment = None

    if status == "Ok":
        environment = response.json().get("environment") or {}
        runtime = environment.get("runtimeVersion")
        default_environment = environment.get("name") or None

    settings_rows.append({
        "WorkspaceId":        workspace_id,
        "WorkspaceName":      ws["WorkspaceName"],
        "CapacityId":         ws["CapacityId"],
        "HasMyAccess":        ws["HasMyAccess"],
        "RuntimeVersion":     runtime,
        "DefaultEnvironment": default_environment,
        "AccessStatus":       status,
        "Detail":             detail,
    })

workspace_runtime_df = pd.DataFrame(settings_rows)

workspace_names = dict(
    zip(workspace_runtime_df["WorkspaceId"], workspace_runtime_df["WorkspaceName"])
)
known_workspace_ids = set(workspace_runtime_df["WorkspaceId"])

unreadable_df = workspace_runtime_df[workspace_runtime_df["AccessStatus"] != "Ok"]

print("\n" + "=" * 62)
print("RUNTIME PAR DEFAUT")
print("=" * 62)
print("Statuts d'acces, avec la cause et l'action correspondante :")

for status, count in workspace_runtime_df["AccessStatus"].value_counts().items():
    meaning, action = STATUS_ACTIONS.get(status, ("Erreur non classee", "Analyser Detail"))
    print(f"  {status:<20} {count:>4}   {meaning}")
    if status != "Ok":
        print(f"  {'':<20} {'':>4}   -> {action}")

print("\nRuntimes lus (None = workspace illisible) :")
print(workspace_runtime_df["RuntimeVersion"].value_counts(dropna=False).to_string())

if len(unreadable_df):
    print(f"\n{len(unreadable_df)} workspaces sans runtime lisible :")
    display(unreadable_df[["WorkspaceName", "HasMyAccess", "AccessStatus", "Detail"]])


# 2. Charge Spark du tenant

Recense les **Environments**, **Notebooks** et **Spark Job Definitions** de tout le tenant via `GET /v1/admin/items`, puis lit le Runtime effectif de chaque Environment.

Ce sont les seuls objets affectés par un changement de Runtime. Un Workspace qui n'en contient aucun n'exécute pas de code Spark.

In [ ]:
# ============================================================
# 2. Environments, Notebooks et Spark Job Definitions
# ============================================================

# ------------------------------------------------------------
# 2.a. Environments et leur runtime effectif
# ------------------------------------------------------------

environment_items = [
    {
        "WorkspaceId":     str(item.get("workspaceId")),
        "EnvironmentId":   str(item.get("id")),
        "EnvironmentName": item.get("displayName") or item.get("name"),
    }
    for item in admin_list_items("Environment")
    if str(item.get("workspaceId")) in known_workspace_ids
]

environment_rows = []

for item in environment_items:

    workspace_id   = item["WorkspaceId"]
    environment_id = item["EnvironmentId"]
    base = f"/v1/workspaces/{workspace_id}/environments/{environment_id}"

    published = api_call("get", f"{base}/sparkcompute", params={"beta": "false"})
    status, detail = describe_status(published)

    published_runtime = None
    staging_runtime = None

    if status == "Ok":
        published_runtime = published.json().get("runtimeVersion")

        staging = api_call("get", f"{base}/staging/sparkcompute", params={"beta": "false"})
        if staging.status_code == 200:
            staging_runtime = staging.json().get("runtimeVersion")

    environment_rows.append({
        "WorkspaceId":      workspace_id,
        "WorkspaceName":    workspace_names.get(workspace_id),
        "EnvironmentId":    environment_id,
        "EnvironmentName":  item["EnvironmentName"],
        "PublishedRuntime": published_runtime,
        "StagingRuntime":   staging_runtime,
        "AccessStatus":     status,
        "Detail":           detail,
    })

environment_df = pd.DataFrame(environment_rows, columns=[
    "WorkspaceId", "WorkspaceName", "EnvironmentId", "EnvironmentName",
    "PublishedRuntime", "StagingRuntime", "AccessStatus", "Detail",
])

if len(environment_df):
    environment_df["EffectiveRuntime"] = environment_df["PublishedRuntime"].fillna(
        environment_df["StagingRuntime"]
    )
    # Un staging different du publie signale une modification jamais publiee.
    environment_df["PendingPublish"] = (
        environment_df["StagingRuntime"].notna()
        & (environment_df["StagingRuntime"] != environment_df["PublishedRuntime"])
    )
else:
    environment_df["EffectiveRuntime"] = None
    environment_df["PendingPublish"] = False

# ------------------------------------------------------------
# 2.b. Notebooks et Spark Job Definitions
# ------------------------------------------------------------

item_rows = []

for item_type in ("Notebook", "SparkJobDefinition"):
    for item in admin_list_items(item_type):

        workspace_id = str(item.get("workspaceId"))

        if workspace_id not in known_workspace_ids:
            continue

        item_rows.append({
            "WorkspaceId":   workspace_id,
            "WorkspaceName": workspace_names.get(workspace_id),
            "ItemId":        str(item.get("id")),
            "ItemName":      item.get("displayName") or item.get("name"),
            "Type":          item_type,
        })

items_df = pd.DataFrame(item_rows, columns=[
    "WorkspaceId", "WorkspaceName", "ItemId", "ItemName", "Type",
])

print("=" * 62)
print("CHARGE SPARK DU TENANT")
print("=" * 62)
print(f"Environments          : {len(environment_df)}")
print(f"Notebooks             : {int((items_df['Type'] == 'Notebook').sum())}")
print(f"Spark Job Definitions : {int((items_df['Type'] == 'SparkJobDefinition').sum())}")

if len(environment_df):
    print("\nRuntime effectif des Environments :")
    print(environment_df["EffectiveRuntime"].value_counts(dropna=False).to_string())

    blocked_env = environment_df[environment_df["AccessStatus"] != "Ok"]
    if len(blocked_env):
        print(f"\n{len(blocked_env)} Environments illisibles faute de permission.")

    pending = environment_df[environment_df["PendingPublish"]]
    if len(pending):
        print(f"\n{len(pending)} Environments ont des changements non publies :")
        display(pending[["WorkspaceName", "EnvironmentName", "PublishedRuntime", "StagingRuntime"]])


# 3. Segmentation d'impact

Répartit les Workspaces à migrer en deux vagues.

| Vague | Définition | Conséquence |
|---|---|---|
| **A** | Aucun Notebook, SJD ni Environment | Migration directe, sans validation |
| **B** | Au moins un objet Spark | Prévenir les Admins et faire tester |

`profile_df` donne, pour chaque Workspace : le Runtime courant, le décompte par type d'objet, la vague et l'éventuel blocage d'accès.

In [ ]:
# ============================================================
# 3. Segmentation des workspaces
# ============================================================

profile_df = workspace_runtime_df.copy()


def _count_by_workspace(frame, mask=None):
    subset = frame if mask is None else frame[mask]
    if not len(subset):
        return {}
    return subset.groupby("WorkspaceId").size().to_dict()


notebook_counts = _count_by_workspace(items_df, items_df["Type"] == "Notebook" if len(items_df) else None)
sjd_counts = _count_by_workspace(items_df, items_df["Type"] == "SparkJobDefinition" if len(items_df) else None)
environment_counts = _count_by_workspace(environment_df)

profile_df["Notebooks"] = profile_df["WorkspaceId"].map(notebook_counts).fillna(0).astype(int)
profile_df["SparkJobDefinitions"] = profile_df["WorkspaceId"].map(sjd_counts).fillna(0).astype(int)
profile_df["Environments"] = profile_df["WorkspaceId"].map(environment_counts).fillna(0).astype(int)

profile_df["SparkObjects"] = (
    profile_df["Notebooks"] + profile_df["SparkJobDefinitions"] + profile_df["Environments"]
)

profile_df["HasSparkWorkload"] = profile_df["SparkObjects"] > 0
profile_df["Wave"] = profile_df["HasSparkWorkload"].map({False: "A", True: "B"})

profile_df["NeedsMigration"] = profile_df["RuntimeVersion"].isin(SOURCE_RUNTIMES)
profile_df["Readable"] = profile_df["AccessStatus"] == "Ok"

# Detection de retrogradation : ne jamais rendre un workspace moins supporte.
profile_df["WouldDowngrade"] = profile_df["RuntimeVersion"].apply(
    lambda v: (runtime_rank(v) is not None) and (TARGET_RANK is not None) and (runtime_rank(v) > TARGET_RANK)
)

downgrades = profile_df[profile_df["WouldDowngrade"]]

if len(downgrades):
    print(f"ATTENTION : {len(downgrades)} workspaces sont sur un runtime plus recent que {TARGET_RUNTIME}.")
    display(downgrades[["WorkspaceName", "RuntimeVersion"]])

    if BLOCK_DOWNGRADE:
        profile_df.loc[profile_df["WouldDowngrade"], "NeedsMigration"] = False
        print("BLOCK_DOWNGRADE actif : ils sont exclus de la migration.\n")

# ------------------------------------------------------------
# Vagues
# ------------------------------------------------------------

to_migrate_df = profile_df[profile_df["NeedsMigration"] & profile_df["Readable"]]

wave_a_df = to_migrate_df[to_migrate_df["Wave"] == "A"]
wave_b_df = to_migrate_df[to_migrate_df["Wave"] == "B"]

# Un workspace illisible a un runtime inconnu : il reste visible, jamais ecarte.
# Mais les causes different, et une seule se resout par un role.
blocked_df = profile_df[~profile_df["Readable"]]

needs_role_df = profile_df[profile_df["AccessStatus"] == "Forbidden"]
no_capacity_df = profile_df[profile_df["AccessStatus"] == "NoFabricCapacity"]
capacity_down_df = profile_df[profile_df["AccessStatus"] == "CapacityUnavailable"]

already_ok_df = profile_df[profile_df["RuntimeVersion"] == TARGET_RUNTIME]

environments_to_migrate_df = environment_df[
    environment_df["EffectiveRuntime"].isin(SOURCE_RUNTIMES)
    & (environment_df["AccessStatus"] == "Ok")
] if len(environment_df) else environment_df

items_affected_df = items_df[
    items_df["WorkspaceId"].isin(set(wave_b_df["WorkspaceId"]))
] if len(items_df) else items_df

print("=" * 62)
print("SEGMENTATION")
print("=" * 62)
print(f"Workspaces retenus                    : {len(profile_df)}")
print(f"  deja en {TARGET_RUNTIME}                        : {len(already_ok_df)}")
print(f"  a migrer et accessibles             : {len(to_migrate_df)}")
print(f"      vague A, sans charge Spark      : {len(wave_a_df)}")
print(f"      vague B, avec charge Spark      : {len(wave_b_df)}")
print(f"  runtime inconnu                     : {len(blocked_df)}")
print(f"      role manquant, elevable         : {len(needs_role_df)}")
print(f"      sans capacite Fabric, hors scope: {len(no_capacity_df)}")
print(f"      capacite en pause, a relancer   : {len(capacity_down_df)}")

print(f"\nObjets a valider en vague B :")
print(f"  Notebooks             : {int(wave_b_df['Notebooks'].sum())}")
print(f"  Spark Job Definitions : {int(wave_b_df['SparkJobDefinitions'].sum())}")
print(f"  Environments          : {int(wave_b_df['Environments'].sum())}")
print(f"  Environments a republier : {len(environments_to_migrate_df)}")

print("\n--- VAGUE A : migration directe ---")
if len(wave_a_df):
    display(wave_a_df[["WorkspaceName", "RuntimeVersion", "SparkObjects"]])
else:
    print("Aucun workspace sans charge Spark a migrer.")

print("\n--- VAGUE B : prevenir et tester ---")
if len(wave_b_df):
    display(
        wave_b_df.sort_values("SparkObjects", ascending=False)[[
            "WorkspaceName", "RuntimeVersion",
            "Notebooks", "SparkJobDefinitions", "Environments", "SparkObjects",
        ]]
    )
else:
    print("Aucun workspace avec charge Spark a migrer.")

if len(needs_role_df):
    print("\n--- ROLE MANQUANT : elevables via AUTO_ELEVATE ---")
    display(needs_role_df[["WorkspaceName", "AccessStatus", "Wave", "SparkObjects"]])

if len(no_capacity_df):
    print("\n--- SANS CAPACITE FABRIC : aucun runtime Spark, rien a migrer ---")
    display(no_capacity_df[["WorkspaceName", "AccessStatus", "SparkObjects", "Detail"]])

if len(capacity_down_df):
    print("\n--- CAPACITE INDISPONIBLE : reprendre la capacite puis relancer ---")
    display(capacity_down_df[["WorkspaceName", "CapacityId", "AccessStatus", "Detail"]])


# 4. Admins à prévenir

Résout les Admins et Membres des Workspaces de la **vague B** via `GET /v1/admin/workspaces/{id}/users`.

`notification_df` est directement exploitable comme liste de diffusion : un destinataire par Workspace, avec le décompte d'objets à tester. Les Workspaces sans Admin identifié sont signalés — ce sont des Workspaces orphelins, à traiter avant migration.

Cette étape consomme **un appel admin par Workspace de la vague B**.

In [ ]:
# ============================================================
# 4. Liste de diffusion pour la vague B
# ============================================================

contacts_df = pd.DataFrame()
notification_df = pd.DataFrame()

target_ids = sorted(set(wave_b_df["WorkspaceId"]))

if not RESOLVE_CONTACTS:
    print("RESOLVE_CONTACTS = False : etape ignoree.")

elif not target_ids:
    print("Vague B vide : aucune notification necessaire.")

else:
    print(f"Resolution des contacts sur {len(target_ids)} workspaces.")
    print(f"Quota admin consomme : {_admin_calls}/{MAX_ADMIN_CALLS}")

    contact_rows = []

    for workspace_id in target_ids:

        response = api_call("get", f"/v1/admin/workspaces/{workspace_id}/users", is_admin=True)
        status, _ = describe_status(response)

        if status != "Ok":
            contact_rows.append({
                "WorkspaceId":   workspace_id,
                "WorkspaceName": workspace_names.get(workspace_id),
                "Role":          None,
                "DisplayName":   None,
                "Principal":     None,
                "Status":        status,
            })
            continue

        for entry in _extract_list(response.json()):

            principal = entry.get("principal") or {}
            user_details = principal.get("userDetails") or {}
            spn_details = principal.get("servicePrincipalDetails") or {}
            access = entry.get("workspaceAccessDetails") or {}

            # L'API expose "workspaceRole", pas "role".
            role = access.get("workspaceRole") or access.get("role")

            principal_type = principal.get("type")

            # Un groupe n'expose pas d'adresse : on retombe sur son nom d'affichage.
            if principal_type == "Group":
                contact = principal.get("displayName") or principal.get("id")
            else:
                contact = (
                    user_details.get("userPrincipalName")
                    or spn_details.get("aadAppId")
                    or principal.get("displayName")
                    or principal.get("id")
                )

            contact_rows.append({
                "WorkspaceId":   workspace_id,
                "WorkspaceName": workspace_names.get(workspace_id),
                "Role":          role,
                "PrincipalType": principal_type,
                "DisplayName":   principal.get("displayName"),
                "Principal":     contact,
                "Status":        "Ok",
            })

    contacts_df = pd.DataFrame(contact_rows)

    notification_df = wave_b_df[[
        "WorkspaceId", "WorkspaceName", "RuntimeVersion",
        "Notebooks", "SparkJobDefinitions", "Environments",
    ]].copy()

    # Si aucun role n'est reconnu alors que des principaux existent, le schema a change.
    resolved = contacts_df[contacts_df["Status"] == "Ok"]

    if len(resolved) and resolved["Role"].isna().all():
        print(
            "\nAVERTISSEMENT : des principaux ont ete lus mais aucun role n'a ete reconnu. "
            "Le schema de l'API a probablement change, verifier la cle du role."
        )
        display(resolved.head(5))

    print("\nRoles trouves :")
    print(resolved["Role"].value_counts(dropna=False).to_string() if len(resolved) else "aucun")

    owners = contacts_df[contacts_df["Role"].isin(NOTIFY_ROLES)]

    if len(owners):
        owners_by_workspace = (
            owners.groupby("WorkspaceId")["Principal"]
            .apply(lambda values: "; ".join(sorted({v for v in values if v})))
            .reset_index(name="Destinataires")
        )
        notification_df = notification_df.merge(owners_by_workspace, on="WorkspaceId", how="left")
    else:
        notification_df["Destinataires"] = None

    notification_df["TargetRuntime"] = TARGET_RUNTIME
    notification_df["Destinataires"] = notification_df["Destinataires"].fillna("AUCUN ADMIN IDENTIFIE")

    orphans = int((notification_df["Destinataires"] == "AUCUN ADMIN IDENTIFIE").sum())

    print(f"\nRoles retenus pour la diffusion : {NOTIFY_ROLES}")
    print(f"Workspaces a notifier           : {len(notification_df)}")
    print(f"Sans destinataire identifie     : {orphans}")

    if orphans == len(notification_df) and len(notification_df):
        print(
            "\nTOUS les workspaces sont sans destinataire : c'est anormal. "
            "Verifier les statuts d'appel ci-dessus et la repartition des roles."
        )
        print(contacts_df["Status"].value_counts().to_string())

    elif orphans:
        print("Ces workspaces sont orphelins : clarifier la propriete avant de migrer.")

    display(notification_df)


# 5. Migration

Déroulé en quatre phases :

| Phase | Action |
|---|---|
| **0** | Élévation *Admin* sur les Workspaces sans accès, attente de propagation, puis re-lecture |
| **1** | Environments : mise à jour du Runtime puis publication, suivie jusqu'à son état terminal |
| **2** | Workspaces : mise à jour du Runtime par défaut, puis relecture de contrôle |
| **3** | Retrait des rôles accordés, exécuté même en cas d'échec |

Le filtrage d'accès n'intervient qu'**après** la phase 0, sinon les Workspaces à élever seraient exclus avant d'avoir été élevés.

**Deux garde-fous cumulatifs** : `DRY_RUN = False` **et** `CONFIRMATION` exacte. Sans les deux, seul le plan est produit.

Seuls les rôles **accordés par le notebook** sont retirés : un accès légitime préexistant n'est jamais révoqué.

In [ ]:
# ============================================================
# 5. Migration
# ============================================================

operations = []
snapshots = []
granted_roles = {}


def log_operation(scope, name, workspace, operation, status,
                  runtime_before=None, runtime_after=None, detail=None):
    operations.append({
        "Scope":         scope,
        "Name":          name,
        "WorkspaceName": workspace,
        "Operation":     operation,
        "Status":        status,
        "RuntimeBefore": runtime_before,
        "RuntimeAfter":  runtime_after,
        "Detail":        detail,
    })


# ------------------------------------------------------------
# Selection de la vague
# ------------------------------------------------------------

if MIGRATION_WAVE == "A":
    selected_df = wave_a_df
elif MIGRATION_WAVE == "B":
    selected_df = wave_b_df
else:
    selected_df = to_migrate_df

selected_ids = set(selected_df["WorkspaceId"])

# Une vague A n'a par definition aucun Environment.
if MIGRATION_WAVE == "A":
    selected_environments = environments_to_migrate_df.iloc[0:0]
else:
    selected_environments = environments_to_migrate_df[
        environments_to_migrate_df["WorkspaceId"].isin(selected_ids)
    ] if len(environments_to_migrate_df) else environments_to_migrate_df

# Seul un 403 se resout par un role : elever sur une capacite absente ou en pause est inutile.
pending_elevation = needs_role_df[needs_role_df["Wave"].isin(
    ["A", "B"] if MIGRATION_WAVE == "ALL" else [MIGRATION_WAVE]
)] if len(needs_role_df) else needs_role_df

expected_operations = len(selected_df) + 2 * len(selected_environments)

print("=" * 62)
print("PLAN DE MIGRATION")
print("=" * 62)
print(f"Vague                  : {MIGRATION_WAVE}")
print(f"Runtime                : {sorted(SOURCE_RUNTIMES)} -> {TARGET_RUNTIME}")
print(f"Workspaces a migrer    : {len(selected_df)}   (1 operation chacun)")
print(f"Environments a migrer  : {len(selected_environments)}   (2 operations chacun)")
print(f"Operations attendues   : {expected_operations}")
print(f"En attente d'elevation : {len(pending_elevation)}")

if not WRITE_ENABLED:
    print("\nMode plan : aucune ecriture effectuee.")
    print("Pour executer :")
    print("  1. valider le plan ci-dessus ;")
    print("  2. prevenir les equipes via notification_df si vague B ;")
    print(f"  3. poser DRY_RUN = False et CONFIRMATION = {EXPECTED_CONFIRMATION}")

else:
    fabric_admin = None

    if AUTO_ELEVATE:
        try:
            import sempy.fabric.admin as fabric_admin
        except ImportError:
            print("sempy.fabric.admin indisponible : mettre a jour semantic-link.")

        if fabric_admin and not ELEVATE_PRINCIPAL:
            raise ValueError(
                "AUTO_ELEVATE = True mais ELEVATE_PRINCIPAL est vide : "
                "les workspaces sans role ne seraient pas migres."
            )

    ws_state = profile_df.copy()
    env_state = environment_df.copy()

    try:
        # ========================================================
        # PHASE 0 : elevation puis re-lecture
        # ========================================================

        if fabric_admin and len(pending_elevation):

            print("\n" + "-" * 62)
            print(f"PHASE 0 : elevation sur {len(pending_elevation)} workspaces")
            print("-" * 62)

            for _, ws in pending_elevation.iterrows():
                try:
                    fabric_admin.add_user_to_workspace(
                        user=ELEVATE_PRINCIPAL,
                        role="Admin",
                        principal_type=ELEVATE_PRINCIPAL_TYPE,
                        workspace=ws["WorkspaceId"],
                    )
                    granted_roles[ws["WorkspaceId"]] = ws["WorkspaceName"]
                    log_operation("Workspace", ws["WorkspaceName"], ws["WorkspaceName"],
                                  "Elevate", "Succeeded")

                except Exception as ex:
                    log_operation("Workspace", ws["WorkspaceName"], ws["WorkspaceName"],
                                  "Elevate", "Failed", detail=str(ex)[:300])

            print(f"Roles accordes : {len(granted_roles)}")

            if granted_roles:
                print(f"Attente de propagation : {ELEVATION_WAIT_S}s")
                time.sleep(ELEVATION_WAIT_S)

                newly_readable = []

                for workspace_id, ws_name in granted_roles.items():
                    check = api_call("get", f"/v1/workspaces/{workspace_id}/spark/settings")

                    if check.status_code == 200:
                        block = check.json().get("environment") or {}
                        runtime = block.get("runtimeVersion")

                        mask = ws_state["WorkspaceId"] == workspace_id
                        ws_state.loc[mask, "RuntimeVersion"] = runtime
                        ws_state.loc[mask, "AccessStatus"] = "Ok"
                        ws_state.loc[mask, "Readable"] = True
                        ws_state.loc[mask, "NeedsMigration"] = runtime in SOURCE_RUNTIMES

                        if runtime in SOURCE_RUNTIMES:
                            newly_readable.append(workspace_id)
                    else:
                        log_operation("Workspace", ws_name, ws_name, "ReadAfterElevation",
                                      f"HTTP{check.status_code}",
                                      detail="Role accorde mais runtime illisible")

                selected_ids |= set(newly_readable)
                selected_df = ws_state[ws_state["WorkspaceId"].isin(selected_ids)]

                # Les Environments de ces workspaces etaient illisibles eux aussi.
                # Sans cette relecture, ils resteraient sur l'ancien runtime et
                # continueraient a l'imposer aux notebooks qui leur sont attaches.
                unlocked_environments = []

                for index, env in env_state.iterrows():

                    if env["WorkspaceId"] not in granted_roles:
                        continue

                    base = f"/v1/workspaces/{env['WorkspaceId']}/environments/{env['EnvironmentId']}"
                    published = api_call("get", f"{base}/sparkcompute", params={"beta": "false"})

                    if published.status_code != 200:
                        continue

                    runtime = published.json().get("runtimeVersion")
                    env_state.at[index, "PublishedRuntime"] = runtime
                    env_state.at[index, "EffectiveRuntime"] = runtime
                    env_state.at[index, "AccessStatus"] = "Ok"

                    if runtime in SOURCE_RUNTIMES:
                        unlocked_environments.append(index)

                if unlocked_environments and MIGRATION_WAVE != "A":
                    selected_environments = pd.concat(
                        [selected_environments, env_state.loc[unlocked_environments]]
                    ).drop_duplicates(subset=["EnvironmentId"])

                print(f"Workspaces devenus migrables   : {len(newly_readable)}")
                print(f"Environments devenus migrables : {len(unlocked_environments)}")
                print(f"Operations revisees            : {len(selected_df) + 2 * len(selected_environments)}")

        # ========================================================
        # PHASE 1 : Environments
        # ========================================================

        for _, env in selected_environments.iterrows():

            workspace_id   = env["WorkspaceId"]
            environment_id = env["EnvironmentId"]
            env_name       = env["EnvironmentName"]
            ws_name        = env["WorkspaceName"]
            before         = env["EffectiveRuntime"]

            base = f"/v1/workspaces/{workspace_id}/environments/{environment_id}"

            try:
                current = api_call("get", f"{base}/staging/sparkcompute", params={"beta": "false"})

                if current.status_code == 200:
                    snapshots.append({
                        "Scope":         "Environment",
                        "WorkspaceName": ws_name,
                        "Name":          env_name,
                        "Config":        json.dumps(current.json()),
                    })

                # Payload minimal : PATCH fusionne, pool et proprietes Spark preserves.
                update = api_call(
                    "patch",
                    f"{base}/staging/sparkcompute",
                    params={"beta": "false"},
                    json_body={"runtimeVersion": TARGET_RUNTIME},
                )

                if update.status_code not in (200, 202):
                    log_operation("Environment", env_name, ws_name, "UpdateRuntime",
                                  f"HTTP{update.status_code}", before,
                                  detail=update.text[:300])
                    continue

                log_operation("Environment", env_name, ws_name, "UpdateRuntime",
                              "Succeeded", before, TARGET_RUNTIME)

                publish = api_call("post", f"{base}/staging/publish", params={"beta": "false"})
                status, error = wait_for_operation(publish, label=f"publish {env_name}")

                if status != "Succeeded":
                    # Cas frequent : conflit de bibliotheques ou JAR incompatible.
                    log_operation("Environment", env_name, ws_name, "Publish",
                                  status, before, detail=error)
                    continue

                published = api_call("get", f"{base}/sparkcompute", params={"beta": "false"})
                after = published.json().get("runtimeVersion") if published.status_code == 200 else None

                log_operation("Environment", env_name, ws_name, "Publish",
                              "Succeeded" if after == TARGET_RUNTIME else "Failed",
                              before, after)

            except Exception as ex:
                log_operation("Environment", env_name, ws_name, "Publish",
                              "Exception", before, detail=str(ex)[:300])

        # ========================================================
        # PHASE 2 : Workspaces
        # ========================================================

        for _, ws in selected_df.iterrows():

            workspace_id = ws["WorkspaceId"]
            ws_name      = ws["WorkspaceName"]
            before       = ws["RuntimeVersion"]

            try:
                snapshots.append({
                    "Scope":         "Workspace",
                    "WorkspaceName": ws_name,
                    "Name":          ws_name,
                    "Config":        json.dumps({"runtimeVersion": before}),
                })

                update = api_call(
                    "patch",
                    f"/v1/workspaces/{workspace_id}/spark/settings",
                    json_body={"environment": {"runtimeVersion": TARGET_RUNTIME}},
                )

                if update.status_code not in (200, 202):
                    log_operation("Workspace", ws_name, ws_name, "UpdateRuntime",
                                  f"HTTP{update.status_code}", before,
                                  detail=update.text[:300])
                    continue

                check = api_call("get", f"/v1/workspaces/{workspace_id}/spark/settings")
                after = (check.json().get("environment") or {}).get("runtimeVersion") \
                    if check.status_code == 200 else None

                log_operation("Workspace", ws_name, ws_name, "UpdateRuntime",
                              "Succeeded" if after == TARGET_RUNTIME else "Failed",
                              before, after)

            except Exception as ex:
                log_operation("Workspace", ws_name, ws_name, "UpdateRuntime",
                              "Exception", before, detail=str(ex)[:300])

    finally:
        # ========================================================
        # PHASE 3 : retrait des roles accordes
        # ========================================================

        if granted_roles and REVOKE_AFTER_MIGRATION:

            print("\n" + "-" * 62)
            print(f"PHASE 3 : retrait de {len(granted_roles)} roles")
            print("-" * 62)

            for workspace_id, ws_name in list(granted_roles.items()):
                try:
                    fabric_admin.delete_user_from_workspace(
                        user=ELEVATE_PRINCIPAL,
                        workspace=workspace_id,
                        is_group=(ELEVATE_PRINCIPAL_TYPE == "Group"),
                    )
                    log_operation("Workspace", ws_name, ws_name, "Revoke", "Succeeded")

                except Exception as ex:
                    log_operation("Workspace", ws_name, ws_name, "Revoke", "Failed",
                                  detail=str(ex)[:300])

        elif granted_roles:
            for ws_name in granted_roles.values():
                log_operation("Workspace", ws_name, ws_name, "Revoke", "Skipped",
                              detail="REVOKE_AFTER_MIGRATION = False")

operations_df = pd.DataFrame(operations, columns=[
    "Scope", "Name", "WorkspaceName", "Operation", "Status",
    "RuntimeBefore", "RuntimeAfter", "Detail",
])
snapshots_df = pd.DataFrame(snapshots, columns=["Scope", "WorkspaceName", "Name", "Config"])

print(f"\nOperations enregistrees : {len(operations_df)}")


# 6. Compte rendu de synthèse

Consolide l'état final : périmètre, segmentation, objets migrés, échecs, rôles restants et Workspaces non traités.

`summary_df` est le tableau de synthèse à archiver. `operations_df` conserve la trace détaillée, `snapshots_df` la configuration antérieure pour un éventuel retour arrière.

> Runtime 1.2 étant hors support, un retour arrière n'est pas une stratégie durable : il ne fait que différer la migration.

In [ ]:
# ============================================================
# 6. Compte rendu
# ============================================================

done = operations_df[operations_df["Status"] == "Succeeded"] if len(operations_df) else operations_df

migrated_workspaces = done[
    (done["Scope"] == "Workspace") & (done["Operation"] == "UpdateRuntime")
] if len(done) else done

migrated_environments = done[
    (done["Scope"] == "Environment") & (done["Operation"] == "Publish")
] if len(done) else done

failures = operations_df[
    ~operations_df["Status"].isin(["Succeeded", "Skipped"])
] if len(operations_df) else operations_df

leftover = operations_df[
    (operations_df["Operation"] == "Revoke") & (operations_df["Status"] != "Succeeded")
] if len(operations_df) else operations_df

summary_rows = [
    ("Workspaces dans le tenant",          len(all_workspaces_df)),
    ("Workspaces retenus",                 len(profile_df)),
    ("  deja sur la cible",                len(already_ok_df)),
    ("  a migrer, accessibles",            len(to_migrate_df)),
    ("  runtime inconnu",                  len(blocked_df)),
    ("      role manquant",                 len(needs_role_df)),
    ("      sans capacite Fabric",          len(no_capacity_df)),
    ("      capacite indisponible",         len(capacity_down_df)),
    ("Vague A, sans charge Spark",         len(wave_a_df)),
    ("Vague B, avec charge Spark",         len(wave_b_df)),
    ("Notebooks en vague B",               int(wave_b_df["Notebooks"].sum()) if len(wave_b_df) else 0),
    ("Spark Job Definitions en vague B",   int(wave_b_df["SparkJobDefinitions"].sum()) if len(wave_b_df) else 0),
    ("Environments en vague B",            int(wave_b_df["Environments"].sum()) if len(wave_b_df) else 0),
    ("Workspaces migres",                  len(migrated_workspaces)),
    ("Environments republies",             len(migrated_environments)),
    ("Operations en echec",                len(failures)),
    ("Roles restant accordes",             len(leftover)),
]

summary_df = pd.DataFrame(summary_rows, columns=["Indicateur", "Valeur"])

print("=" * 62)
print(f"COMPTE RENDU - cible Runtime {TARGET_RUNTIME} - vague {MIGRATION_WAVE}")
print("=" * 62)
print(summary_df.to_string(index=False))

if not WRITE_ENABLED:
    print("\nExecution en mode plan : aucune modification n'a ete appliquee.")

# ------------------------------------------------------------
# Repartition des runtimes, avant et apres
# ------------------------------------------------------------

before_counts = profile_df["RuntimeVersion"].value_counts(dropna=False).rename("Avant")

after_series = profile_df.set_index("WorkspaceId")["RuntimeVersion"].copy()

if len(migrated_workspaces):
    migrated_names = set(migrated_workspaces["WorkspaceName"])
    mask = profile_df["WorkspaceName"].isin(migrated_names)
    after_series.loc[profile_df.loc[mask, "WorkspaceId"]] = TARGET_RUNTIME

after_counts = after_series.value_counts(dropna=False).rename("Apres")

runtime_shift_df = pd.concat([before_counts, after_counts], axis=1).fillna(0).astype(int)

print("\nREPARTITION DES RUNTIMES")
print(runtime_shift_df.to_string())

# ------------------------------------------------------------
# Points d'attention
# ------------------------------------------------------------

if len(failures):
    print(f"\nECHECS : {len(failures)}")
    print("Un echec de publication d'Environment vient le plus souvent d'un conflit")
    print("de bibliotheques ou d'un JAR incompatible avec le nouveau runtime.")
    display(failures[["Scope", "Name", "WorkspaceName", "Operation", "Status", "Detail"]])

if len(leftover):
    print(f"\nATTENTION : {len(leftover)} roles restent accordes. Les retirer manuellement.")
    display(leftover[["WorkspaceName", "Status", "Detail"]])

if len(blocked_df):
    print(f"\nNON TRAITES : {len(blocked_df)}")
    print("  Forbidden           -> attribuer un role, puis relancer")
    print("  NoFabricCapacity    -> hors perimetre, aucun runtime Spark")
    print("  CapacityUnavailable -> reprendre la capacite, puis relancer")
    display(blocked_df[["WorkspaceName", "Wave", "AccessStatus", "SparkObjects", "Detail"]])

remaining = profile_df[
    profile_df["NeedsMigration"]
    & ~profile_df["WorkspaceName"].isin(set(migrated_workspaces["WorkspaceName"]) if len(migrated_workspaces) else set())
]

if len(remaining):
    print(f"\nRESTE A MIGRER : {len(remaining)}")
    print(remaining.groupby("Wave").size().to_string())

if len(snapshots_df):
    print(f"\n{len(snapshots_df)} snapshots de configuration conserves dans snapshots_df.")

print("\nProchaine etape :")
if MIGRATION_WAVE == "A" and len(wave_b_df):
    print(f"  Prevenir les {len(wave_b_df)} workspaces de la vague B via notification_df,")
    print("  faire valider les traitements, puis relancer avec MIGRATION_WAVE = B.")
elif len(blocked_df):
    print(f"  Obtenir un role sur les {len(blocked_df)} workspaces bloques, puis relancer.")
else:
    print("  Aucun reste identifie sur ce perimetre.")

display(summary_df)
